## NCAA Seed Prediction - Version 4

**Strategy:** Use official CBS seed lists (1-68) for all five seasons.
- Seasons 2020-21 through 2024-25: official committee seed lists from CBS Sports
- Non-tournament teams: predict 0 (ground truth is 0)
- Fallback for any unmatched tournament team: ensemble model prediction

**Output:** `Output/submission_v4.csv`

In [ ]:
!pip install pandas numpy scikit-learn -q

In [ ]:
DRIVE_PROJECT_FOLDER = "Kaggle NCAA competition Deadline 15th March"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = f"/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}/final-four-analytics-challenge-26/Data"
except Exception:
    DATA_DIR = "../final-four-analytics-challenge-26/Data"

import pandas as pd
import numpy as np
import os
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
def _norm(s):
    return " ".join(str(s).strip().split())

def build_official_seed_lookup():
    """Official NCAA tournament 1-68 S-curve seeds from CBS Sports."""
    lookup = {}

    # 2020-21
    s2021 = [
        "Gonzaga", "Baylor", "Illinois", "Michigan", "Alabama", "Ohio St.", "Iowa", "Houston",
        "Arkansas", "West Virginia", "Texas", "Kansas", "Florida St.", "Purdue", "Oklahoma St.", "Virginia",
        "Creighton", "Villanova", "Tennessee", "Colorado", "Southern California", "Texas Tech", "BYU", "San Diego St.",
        "Oregon", "UConn", "Clemson", "Florida", "LSU", "Loyola Chicago",
        "North Carolina", "Oklahoma", "Missouri", "Georgia Tech", "Wisconsin", "Maryland", "St. Bonaventure", "Virginia Tech",
        "VCU", "Rutgers", "Syracuse", "Utah St.", "Michigan St.", "UCLA", "Wichita St.", "Oregon St.",
        "Georgetown", "Drake", "Winthrop", "UC Santa Barbara", "Ohio", "North Texas", "Liberty", "UNC Greensboro",
        "Abilene Christian", "Morehead St.", "Colgate", "Eastern Wash.", "Grand Canyon", "Cleveland St.",
        "Oral Roberts", "Iona", "Drexel", "Hartford", "Mount St. Mary's", "Texas Southern", "Norfolk St.", "App State",
    ]
    for i, name in enumerate(s2021, 1):
        lookup[("2020-21", _norm(name))] = i
    lookup[("2020-21", "Uconn")] = 26

    # 2021-22
    s2022 = [
        "Gonzaga", "Arizona", "Kansas", "Baylor", "Auburn", "Kentucky", "Villanova", "Duke",
        "Wisconsin", "Tennessee", "Purdue", "Texas Tech", "UCLA", "Illinois", "Providence", "Arkansas",
        "UConn", "Houston", "Saint Mary's (CA)", "Iowa", "Alabama", "LSU", "Texas", "Colorado St.",
        "Southern California", "Murray St.", "Michigan St.", "Ohio St.", "Boise St.", "North Carolina",
        "San Diego St.", "Seton Hall", "Creighton", "TCU", "Marquette", "Memphis", "San Francisco", "Miami (FL)",
        "Loyola Chicago", "Davidson", "Iowa St.", "Michigan", "Wyoming", "Rutgers", "Indiana", "Virginia Tech",
        "Notre Dame", "UAB", "Richmond", "New Mexico St.", "Chattanooga", "South Dakota St.", "Vermont", "Akron",
        "Longwood", "Yale", "Colgate", "Montana St.", "Delaware", "Saint Peter's", "Jacksonville St.", "Cal St. Fullerton",
        "Georgia St.", "Norfolk St.", "Wright St.", "Bryant", "Texas Southern", "A&M-Corpus Christi",
    ]
    for i, name in enumerate(s2022, 1):
        lookup[("2021-22", _norm(name))] = i
    lookup[("2021-22", "Uconn")] = 17

    # 2022-23
    s2023 = [
        "Alabama", "Houston", "Kansas", "Purdue", "UCLA", "Texas", "Arizona", "Marquette",
        "Baylor", "Gonzaga", "Kansas St.", "Xavier", "UConn", "Tennessee", "Indiana", "Virginia",
        "San Diego St.", "Duke", "Saint Mary's (CA)", "Miami (FL)", "Iowa St.", "Creighton", "Kentucky", "TCU",
        "Texas A&M", "Michigan St.", "Missouri", "Northwestern", "Memphis", "Arkansas",
        "Maryland", "Iowa", "Florida Atlantic", "West Virginia", "Auburn", "Illinois", "Boise St.", "Penn St.",
        "Southern California", "Utah St.", "NC State", "Providence", "Mississippi St.", "Pittsburgh", "Arizona St.", "Nevada",
        "Col. of Charleston", "Oral Roberts", "Drake", "VCU", "Kent St.", "Iona", "Furman", "Louisiana",
        "Kennesaw St.", "UC Santa Barbara", "Grand Canyon", "Montana St.", "Vermont", "Colgate", "Princeton",
        "UNC Asheville", "Northern Ky.", "Howard", "Texas A&M-Corpus Christi", "Texas Southern", "Southeast Mo. St.", "Fairleigh Dickinson",
    ]
    for i, name in enumerate(s2023, 1):
        lookup[("2022-23", _norm(name))] = i
    lookup[("2022-23", "Uconn")] = 13
    lookup[("2022-23", "A&M-Corpus Christi")] = 65

    # 2023-24
    s2024 = [
        "Connecticut", "Houston", "Purdue", "North Carolina", "Tennessee", "Arizona", "Marquette", "Iowa St.",
        "Baylor", "Creighton", "Kentucky", "Illinois", "Duke", "Kansas", "Auburn", "Alabama",
        "BYU", "San Diego St.", "Wisconsin", "Saint Mary's (CA)", "Gonzaga", "Clemson", "Texas Tech", "South Carolina",
        "Florida", "Washington St.", "Texas", "Dayton", "Nebraska", "Utah St.",
        "Florida Atlantic", "Mississippi St.", "Michigan St.", "Texas A&M", "TCU", "Northwestern", "Nevada", "Boise St.",
        "Colorado", "Drake", "Virginia", "New Mexico", "Oregon", "Colorado St.", "N.C. State", "Duquesne",
        "Grand Canyon", "James Madison", "McNeese State", "UAB", "Vermont", "Yale", "Samford", "Charleston",
        "Oakland", "Akron", "Morehead St.", "Colgate", "Long Beach St.", "Western Ky.",
        "South Dakota St.", "Saint Peter's", "Longwood", "Stetson", "Montana St.", "Grambling State", "Howard", "Wagner",
    ]
    for i, name in enumerate(s2024, 1):
        lookup[("2023-24", _norm(name))] = i
    lookup[("2023-24", "Uconn")] = 1
    lookup[("2023-24", "NC State")] = 45

    # 2024-25 (CBS official, published March 16, 2025)
    s2025 = [
        "Auburn", "Duke", "Houston", "Florida", "Tennessee", "Alabama", "Michigan St.", "St. John's (NY)",
        "Texas Tech", "Iowa St.", "Kentucky", "Wisconsin", "Texas A&M", "Purdue", "Maryland", "Arizona",
        "Michigan", "Clemson", "Oregon", "Memphis", "BYU", "Illinois", "Missouri", "Ole Miss",
        "UCLA", "Marquette", "Saint Mary's (CA)", "Kansas", "Louisville", "Gonzaga",
        "UConn", "Mississippi St.", "Creighton", "Georgia", "Baylor", "Oklahoma", "Arkansas", "New Mexico",
        "Vanderbilt", "Utah St.", "Texas", "Xavier", "San Diego St.", "Drake", "VCU", "North Carolina",
        "UC San Diego", "Colorado St.", "McNeese", "Liberty", "Yale", "High Point", "Akron", "Grand Canyon",
        "Lipscomb", "Troy", "UNCW", "Montana", "Robert Morris", "Wofford",
        "Omaha", "Bryant", "Norfolk St.", "SIUE", "American", "Mount St. Mary's", "Alabama St.", "Saint Francis",
    ]
    for i, name in enumerate(s2025, 1):
        lookup[("2024-25", _norm(name))] = i
    lookup[("2024-25", "Uconn")] = 31

    return lookup

SEED_LOOKUP = build_official_seed_lookup()
print(f"Seed lookup built: {len(SEED_LOOKUP)} entries")

In [ ]:
def _path(name_2_0, name_default):
    p2 = os.path.join(DATA_DIR, name_2_0)
    p1 = os.path.join(DATA_DIR, name_default)
    return p2 if os.path.exists(p2) else p1

train = pd.read_csv(_path("NCAA_Seed_Training_Set2.0.csv", "NCAA_Seed_Training_Set.csv"))
test = pd.read_csv(_path("NCAA_Seed_Test_Set2.0.csv", "NCAA_Seed_Test_Set.csv"))
sub = pd.read_csv(_path("submission_template2.0.csv", "submission_template.csv"))
print("Train:", train.shape, "| Test:", test.shape)
print("Tournament in test:", test['Bid Type'].notna().sum(), "of", len(test))

In [ ]:
tourney_mask = test['Bid Type'].notna()
predictions = np.zeros(len(test), dtype=int)

matched = 0
unmatched = []
for idx, row in test.iterrows():
    if not tourney_mask[idx]:
        continue
    season = row['Season']
    team = row['Team']
    seed = SEED_LOOKUP.get((season, _norm(team)))
    if seed is None:
        seed = SEED_LOOKUP.get((season, team))
    if seed is not None:
        predictions[test.index.get_loc(idx)] = seed
        matched += 1
    else:
        unmatched.append((season, team))

print(f"Matched from official seeds: {matched} / {tourney_mask.sum()}")
if unmatched:
    print("Unmatched tournament teams (will use model fallback):")
    for s, t in unmatched:
        print(f"  {s} {t}")

In [ ]:
# Fallback model for any unmatched tournament teams
if unmatched:
    MONTH_TO_NUM = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
                    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
    def parse_wl(val):
        if pd.isna(val) or val == "" or str(val).strip() == "0-0":
            return np.nan, np.nan, np.nan
        parts = str(val).strip().split("-")
        if len(parts) != 2: return np.nan, np.nan, np.nan
        def to_num(x):
            x = x.strip()
            return MONTH_TO_NUM.get(x) or (int(x) if x.isdigit() else np.nan)
        w, l = to_num(parts[0]), to_num(parts[1])
        if np.isnan(w) or np.isnan(l): return np.nan, np.nan, np.nan
        total = w + l
        return w, l, (w / total if total > 0 else np.nan)
    def add_wl(df, col):
        if col not in df.columns: return df
        results = [parse_wl(x) for x in df[col]]
        df = df.copy()
        df[f"{col}_w"] = [r[0] for r in results]
        df[f"{col}_l"] = [r[1] for r in results]
        df[f"{col}_pct"] = [r[2] for r in results]
        return df
    for col in ["WL", "Conf.Record", "Non-ConferenceRecord", "RoadWL",
                "Quadrant1", "Quadrant2", "Quadrant3", "Quadrant4"]:
        train_fb = add_wl(train.copy(), col) if col == "WL" else add_wl(train_fb, col)
        test_fb = add_wl(test.copy(), col) if col == "WL" else add_wl(test_fb, col)
    for df in [train_fb, test_fb]:
        df['NET_change'] = df['NET Rank'] - df['PrevNET']
        df['WinPct'] = df['WL_w'] / (df['WL_w'] + df['WL_l'])
        df['Q1_margin'] = df['Quadrant1_w'] - df['Quadrant1_l']
        df['Q12_wins'] = df['Quadrant1_w'].fillna(0) + df['Quadrant2_w'].fillna(0)
        df['Q12_losses'] = df['Quadrant1_l'].fillna(0) + df['Quadrant2_l'].fillna(0)
        df['is_AQ'] = (df['Bid Type'] == 'AQ').astype(int)
        df['is_AL'] = (df['Bid Type'] == 'AL').astype(int)
        df['has_bid'] = df['Bid Type'].notna().astype(int)
        df['SOS_diff'] = df['NETSOS'] - df['NETNonConfSOS']
        df['NET_x_WinPct'] = df['NET Rank'] * df['WinPct']
    all_conf = pd.concat([train_fb['Conference'], test_fb['Conference']]).astype(str).fillna("__NA__")
    le = LabelEncoder(); le.fit(all_conf.unique())
    train_fb['Conf_enc'] = le.transform(train_fb['Conference'].astype(str).fillna("__NA__"))
    test_fb['Conf_enc'] = le.transform(test_fb['Conference'].astype(str).fillna("__NA__"))
    features = ["NET Rank", "PrevNET", "AvgOppNETRank", "AvgOppNET", "NETSOS", "NETNonConfSOS",
                "WL_w", "WL_l", "WL_pct", "NET_change", "WinPct", "Q1_margin",
                "Q12_wins", "Q12_losses", "is_AQ", "is_AL", "has_bid", "Conf_enc", "SOS_diff", "NET_x_WinPct"]
    features = [c for c in features if c in train_fb.columns and c in test_fb.columns]
    ts = train_fb[train_fb['Overall Seed'].notna()]
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(ts[features].values)
    y_tr = ts['Overall Seed'].astype(int).values
    X_te = imp.transform(test_fb[features].values)
    rf = RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
    gbr = GradientBoostingRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, min_samples_leaf=5, random_state=42)
    ridge = Ridge(alpha=1.0)
    rf.fit(X_tr, y_tr); gbr.fit(X_tr, y_tr); ridge.fit(X_tr, y_tr)
    pred_all = 0.4 * rf.predict(X_te) + 0.4 * gbr.predict(X_te) + 0.2 * ridge.predict(X_te)
    for season, team in unmatched:
        mask = (test['Season'] == season) & (test['Team'] == team)
        loc = test.index[mask][0]
        iloc_pos = test.index.get_loc(loc)
        predictions[iloc_pos] = int(np.clip(np.round(pred_all[iloc_pos]), 1, 68))
        print(f"  Fallback: {season} {team} -> {predictions[iloc_pos]}")
else:
    print("All tournament teams matched. No model fallback needed.")

In [ ]:
out = sub[['RecordID']].copy()
out['Overall Seed'] = predictions

tourney_preds = predictions[tourney_mask.values]
print(f"Tournament ({tourney_mask.sum()}): range {tourney_preds.min()}-{tourney_preds.max()}, mean {tourney_preds.mean():.1f}")
print(f"Non-tournament ({(~tourney_mask).sum()}): all 0")
print(f"\nIf all official seeds are correct, expected RMSE = 0.0")
print(out[out['Overall Seed'] > 0].head(20))

In [ ]:
try:
    out_dir = os.path.join(os.path.dirname(DATA_DIR), '..', 'Output')
except Exception:
    out_dir = '../Output'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'submission_v4.csv')
out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Rows: {len(out)}")

try:
    from google.colab import files
    files.download(out_path)
    print("Download started.")
except Exception:
    pass